# P1 · Quality/song-balanced sampling experiment
Baseline controls are preserved. This notebook runs only the sampling experiment. Enable Internet and GPU.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json
project = Path("/kaggle/working/chord-practice-system")
if not project.exists():
    subprocess.run(["git", "clone", "https://github.com/PhatLavar/chord-practice-system.git", str(project)], check=True)
os.chdir(project)
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run([sys.executable,"-m","pip","install","--no-deps","-e","."],check=True)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True))
import torch, numpy
print("torch", torch.__version__, "numpy", numpy.__version__, "CUDA", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Official inputs. Downloaded archives are read without extraction.
from urllib.request import urlretrieve
raw = project / "data/raw"
raw.mkdir(parents=True, exist_ok=True)
sources = {
    "index.csv": "https://www.dropbox.com/s/o0olz0uwl9z9stb/billboard-2.0-index.csv?dl=1",
    "full.tar.xz": "https://www.dropbox.com/s/t390alzrkx0c9yt/billboard-2.0.1-lab.tar.xz?dl=1",
    "billboard-2.0-chordino.tar.xz": "https://www.dropbox.com/s/e9dm23vbawg9dsw/billboard-2.0-chordino.tar.xz?dl=1",
}
for name, url in sources.items():
    if not (raw / name).exists():
        temporary = raw / (name + ".part")
        urlretrieve(url, temporary)
        temporary.replace(raw / name)
cache = project / "data/processed/p1"
if not cache.exists():
    subprocess.run([sys.executable, '-m', 'acr.data.billboard.prepare'], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"], check=True)


In [ ]:
balanced_run = project / "runs/p1-balanced-kaggle"
assert torch.cuda.is_available(), "Enable a Kaggle GPU before this training run."
subprocess.run([sys.executable, '-m', 'acr.training.frame',
                "--config", "configs/p1/p1-balanced.json",
                "--output", str(balanced_run), "--device", "cuda"], check=True)
print((balanced_run / "metrics.json").read_text())
print((balanced_run / "sampling.json").read_text())


In [ ]:
subprocess.run([sys.executable, '-m', 'acr.evaluation.frame',
                "--run", str(balanced_run),
                "--output", str(balanced_run / "diagnostics")], check=True)
print((balanced_run / "diagnostics/summary.json").read_text())


In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/p1-balanced-results", "zip",
                              project, "runs/p1-balanced-kaggle")
print(archive)
